In [1]:
import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


In [2]:
master = pd.read_csv(
    'cleaned_master_dataset.csv'
)

master.shape

(82365, 39)

# Create Customer-Level Dataset

In [3]:
customer_cluster = (
    master
    .groupby('customer_unique_id')
    .agg({
        'revenue':'sum',
        'order_id':'nunique'
    })
    .reset_index()
)

customer_cluster.columns = [
    'customer_unique_id',
    'Total_Revenue',
    'Total_Orders'
]

customer_cluster.head()

,customer_unique_id,Total_Revenue,Total_Orders
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1
1,0000f46a3911fa3c0805444483337064,69.0,1
2,0004aac84e0df4da2b147fca70cf8255,0.0,1
3,00053a61a98854899e70ed204dd4bafe,382.0,1
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,1


# Create Average Order Value

In [4]:
customer_cluster['Average_Order_Value'] = (
    customer_cluster['Total_Revenue']
    /
    customer_cluster['Total_Orders']
)

customer_cluster.head()

,customer_unique_id,Total_Revenue,Total_Orders,Average_Order_Value
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1,0.0
1,0000f46a3911fa3c0805444483337064,69.0,1,69.0
2,0004aac84e0df4da2b147fca70cf8255,0.0,1,0.0
3,00053a61a98854899e70ed204dd4bafe,382.0,1,382.0
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,1,104.9


# Check Shape

In [5]:
customer_cluster.shape

(69127, 4)

# Features

In [6]:
features = customer_cluster[
    [
        'Total_Revenue',
        'Total_Orders',
        'Average_Order_Value'
    ]
]

features.head()

,Total_Revenue,Total_Orders,Average_Order_Value
0,0.0,1,0.0
1,69.0,1,69.0
2,0.0,1,0.0
3,382.0,1,382.0
4,104.9,1,104.9


# Scale Features

In [7]:
scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    features
)

scaled_features[:5]

array([[-0.51783264, -0.14090866, -0.51806858],
       [-0.2173441 , -0.14090866, -0.21050665],
       [-0.51783264, -0.14090866, -0.51806858],
       [ 1.14574158, -0.14090866,  1.18466558],
       [-0.06100296, -0.14090866, -0.0504853 ]])

# Apply KMeans

In [8]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

customer_cluster['Cluster'] = (
    kmeans.fit_predict(
        scaled_features
    )
)

customer_cluster.head()

,customer_unique_id,Total_Revenue,Total_Orders,Average_Order_Value,Cluster
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1,0.0,0
1,0000f46a3911fa3c0805444483337064,69.0,1,69.0,0
2,0004aac84e0df4da2b147fca70cf8255,0.0,1,0.0,0
3,00053a61a98854899e70ed204dd4bafe,382.0,1,382.0,1
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,1,104.9,0


# Check Cluster Distribution

In [9]:
customer_cluster[
    'Cluster'
].value_counts()

,count
Cluster,
0,62833
1,4310
2,1580
3,404


# Cluster Summary

In [10]:
cluster_summary = (
    customer_cluster
    .groupby('Cluster')
    [
        [
            'Total_Revenue',
            'Total_Orders',
            'Average_Order_Value'
        ]
    ]
    .mean()
)

cluster_summary

,Total_Revenue,Total_Orders,Average_Order_Value
Cluster,,,
0,75.949204,1.000000,75.949204
1,530.676332,1.000000,530.676332
2,205.474082,2.096835,98.281760
3,2068.722748,1.027228,2029.076658


# Assign Names

In [11]:
cluster_names = {
    0:'Budget Customers',
    1:'Regular Customers',
    2:'Premium Customers',
    3:'VIP Customers'
}

customer_cluster[
    'Cluster_Name'
] = customer_cluster[
    'Cluster'
].map(
    cluster_names
)

customer_cluster.head()

,customer_unique_id,Total_Revenue,Total_Orders,Average_Order_Value,Cluster,Cluster_Name
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1,0.0,0,Budget Customers
1,0000f46a3911fa3c0805444483337064,69.0,1,69.0,0,Budget Customers
2,0004aac84e0df4da2b147fca70cf8255,0.0,1,0.0,0,Budget Customers
3,00053a61a98854899e70ed204dd4bafe,382.0,1,382.0,1,Regular Customers
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,1,104.9,0,Budget Customers


# Segment Distribution

In [12]:
customer_cluster[
    'Cluster_Name'
].value_counts()

,count
Cluster_Name,
Budget Customers,62833
Regular Customers,4310
Premium Customers,1580
VIP Customers,404


In [13]:
customer_cluster.to_csv(
    'customer_clusters.csv',
    index=False
)

print("Customer Clustering Completed")

Customer Clustering Completed


In [14]:
import os

os.listdir()

['.config',
 'cleaned_master_dataset.csv',
 'customer_clusters.csv',
 'sample_data']

# Customer Recency Feature Engineering

In [16]:
master['order_purchase_timestamp'] = pd.to_datetime(
    master['order_purchase_timestamp'],
    format='mixed',
    dayfirst=True
)

snapshot_date = (
    master['order_purchase_timestamp']
    .max()
    +
    pd.Timedelta(days=1)
)

recency_df = (
    master.groupby(
        'customer_unique_id'
    )['order_purchase_timestamp']
    .max()
    .reset_index()
)

recency_df['Recency'] = (
    snapshot_date
    -
    recency_df[
        'order_purchase_timestamp'
    ]
).dt.days

recency_df.head()

,customer_unique_id,order_purchase_timestamp,Recency
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:00,161
1,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:00,586
2,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:00,337
3,00053a61a98854899e70ed204dd4bafe,2018-02-28 11:15:00,232
4,0005ef4cd20d2893f0d9fbd94d3c0d97,2018-03-12 15:22:00,220


# Merge Customer Features

In [17]:
customer_cluster = customer_cluster.merge(
    recency_df[
        [
            'customer_unique_id',
            'Recency'
        ]
    ],
    on='customer_unique_id',
    how='left'
)

customer_cluster.head()

,customer_unique_id,Total_Revenue,Total_Orders,Average_Order_Value,Cluster,Cluster_Name,Recency
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1,0.0,0,Budget Customers,161
1,0000f46a3911fa3c0805444483337064,69.0,1,69.0,0,Budget Customers,586
2,0004aac84e0df4da2b147fca70cf8255,0.0,1,0.0,0,Budget Customers,337
3,00053a61a98854899e70ed204dd4bafe,382.0,1,382.0,1,Regular Customers,232
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,1,104.9,0,Budget Customers,220


# Verify Dataset

In [18]:
customer_cluster.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69127 entries, 0 to 69126
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_unique_id   69127 non-null  object 
 1   Total_Revenue        69127 non-null  float64
 2   Total_Orders         69127 non-null  int64  
 3   Average_Order_Value  69127 non-null  float64
 4   Cluster              69127 non-null  int32  
 5   Cluster_Name         69127 non-null  object 
 6   Recency              69127 non-null  int64  
dtypes: float64(2), int32(1), int64(2), object(2)
memory usage: 3.4+ MB


In [20]:
customer_cluster.shape

(69127, 7)

# Feature Selection

In [21]:
features = customer_cluster[
    [
        'Total_Revenue',
        'Total_Orders',
        'Average_Order_Value',
        'Recency'
    ]
]

features.head()

,Total_Revenue,Total_Orders,Average_Order_Value,Recency
0,0.0,1,0.0,161
1,69.0,1,69.0,586
2,0.0,1,0.0,337
3,382.0,1,382.0,232
4,104.9,1,104.9,220


# Feature Scaling

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_features = scaler.fit_transform(
    features
)

scaled_features[:5]

array([[-0.51783264, -0.14090866, -0.51806858, -0.83357668],
       [-0.2173441 , -0.14090866, -0.21050665,  1.93294932],
       [-0.51783264, -0.14090866, -0.51806858,  0.31209056],
       [ 1.14574158, -0.14090866,  1.18466558, -0.3714041 ],
       [-0.06100296, -0.14090866, -0.0504853 , -0.44951778]])

# Elbow Method

In [23]:
from sklearn.cluster import KMeans

inertia = []

for k in range(2,11):

    km = KMeans(
        n_clusters=k,
        random_state=42
    )

    km.fit(
        scaled_features
    )

    inertia.append(
        km.inertia_
    )

print(inertia)

[217755.01662577296, 152022.03567231848, 105518.19903553317, 81703.30288071184, 67186.38748927882, 56702.389325134114, 49475.33928579453, 44614.316001565996, 41160.248361328464]


# KMeans

In [24]:
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

customer_cluster['Cluster'] = (
    kmeans.fit_predict(
        scaled_features
    )
)

customer_cluster.head()

,customer_unique_id,Total_Revenue,Total_Orders,Average_Order_Value,Cluster,Cluster_Name,Recency
0,0000366f3b9a7992bf8c76cfdf3221e2,0.0,1,0.0,3,Budget Customers,161
1,0000f46a3911fa3c0805444483337064,69.0,1,69.0,0,Budget Customers,586
2,0004aac84e0df4da2b147fca70cf8255,0.0,1,0.0,0,Budget Customers,337
3,00053a61a98854899e70ed204dd4bafe,382.0,1,382.0,3,Regular Customers,232
4,0005ef4cd20d2893f0d9fbd94d3c0d97,104.9,1,104.9,3,Budget Customers,220


# Cluster Distribution

In [25]:
customer_cluster[
    'Cluster'
].value_counts()

,count
Cluster,
3,37969
0,28380
2,1577
1,1201


# Revenue Contribution Analysis

In [27]:
cluster_revenue = (
    customer_cluster
    .groupby('Cluster_Name')
    ['Total_Revenue']
    .sum()
    .reset_index()
)

cluster_revenue

,Cluster_Name,Total_Revenue
0,Budget Customers,4772116.34
1,Premium Customers,324649.05
2,Regular Customers,2287214.99
3,VIP Customers,835763.99


# Customer Percentage Analysis

In [28]:
cluster_percentage = (
    customer_cluster[
        'Cluster_Name'
    ]
    .value_counts(normalize=True)
    * 100
)

cluster_percentage

,proportion
Cluster_Name,
Budget Customers,90.895019
Regular Customers,6.234901
Premium Customers,2.285648
VIP Customers,0.584432


In [29]:
customer_cluster.to_csv(
    'customer_clusters.csv',
    index=False
)

cluster_summary.to_csv(
    'cluster_summary.csv'
)

cluster_revenue.to_csv(
    'cluster_revenue.csv',
    index=False
)

print("Customer Clustering Files Saved")

Customer Clustering Files Saved


In [30]:
import os

os.listdir()

['.config',
 'cleaned_master_dataset.csv',
 'customer_clusters.csv',
 'cluster_revenue.csv',
 'cluster_summary.csv',
 'sample_data']